# HCI2000 Data Preprocessing
Browser tab orientation (horizontal / vertical) x tab load (low / high). 16 participants, 24 trials each.

## What this notebook does
Our experiment produced **four separate sources of data**. Before we can analyse anything, we clean each one and then combine them into two tidy tables.

| Source | One row is | Expected rows |
|---|---|---|
| `main_data.csv` (one file per participant) | one trial | 16 x 24 = 384 |
| Block NASA-TLX form (after each condition) | one participant in one condition | 16 x 4 = 64 |
| Weighted NASA-TLX form (15 pairwise questions) | one participant | 16 |
| Screening / background form | one person | 16 (plus anyone who was never enrolled) |

These sources are at **three different levels** (trial, participant-by-condition, participant), which is why we cannot just stack them into one file.
We clean each source on its own, and only then merge them.

## The plan
1. **Load** the four sources.
2. **Clean each source separately** and check that it looks the way the study design says it should.
3. **Check who is present in which source.**
4. **Combine** into two tables: `trial_level` (384 rows) and `condition_level` (64 rows).
5. **Save** them, together with a log of everything we changed.

## Three rules we follow
- **Join by participant code (and condition), never by row order.** Row order is how two forms silently get mixed up.
- **Flag, don't silently delete.** Every change is written to a cleaning log with a count.
- **Check our assumptions with `assert`.** If the data break something the design promises (for example, 24 trials per participant), the notebook stops and tells us.

## 0. Setup

We tell the notebook where the project folder is (change this one line if the folder moves), where the raw files are, and a few facts about the study.
The dummy data use the code prefix `DUMMY` and the mode `test`. For the real data we change these to `P` and `main`.

In [ ]:
import glob
import os
import pandas as pd

# ---- the ONE line to change if the project folder moves ----
BASE = "/Users/jphonrado/Documents/School/Term 1 A.Y. 26-27/HCI2000/HCI2000-Data-Analysis"

RAW_FOLDER = BASE + "/Data Preprocessing/dummy_data"   # where the raw files are
OUT_FOLDER = BASE + "/Data Preprocessing/cleaned"      # where the cleaned files will be saved

PREFIX = "DUMMY"          # participant codes start with this  (real data: "P")
MODE = "test"             # keep only sessions with this mode   (real data: "main")
N_PARTICIPANTS = 16
TRIALS_PER_BLOCK = 6
CONDITIONS = ["H-L", "H-H", "V-L", "V-H"]
DIMS = ["mental_demand", "physical_demand", "temporal_demand", "performance", "effort", "frustration"]

# the participant codes we expect: DUMMY001 ... DUMMY016
ROSTER = [PREFIX + str(n).zfill(3) for n in range(1, N_PARTICIPANTS + 1)]

# a small helper that prints what we changed AND remembers it for the cleaning log
log = []
def note(step, what, how_many):
    log.append({"step": step, "what happened": what, "how many": how_many})
    print(step, "-", what, ":", how_many)

## 1. Load the raw sources
### 1.1 `main_data.csv`: combine the 16 participant files into one table
Each participant has their own file. We read them all and stack them (one below the other).
Before stacking, we check that every file has exactly the same columns, because stacking files with different columns would quietly create gaps.

In [ ]:
files = sorted(glob.glob(RAW_FOLDER + "/main_data/*_main_data.csv"))
print(len(files), "files found")

# every file must have the same column names
column_sets = {tuple(pd.read_csv(f, nrows=0).columns) for f in files}
assert len(column_sets) == 1, "the main_data files do not all have the same columns"

# read every file and stack them into one table
main_raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print("main_data (raw):", main_raw.shape)
main_raw.head(3)

### 1.2 The three forms (block NASA-TLX, weighted NASA-TLX, screening)

In [ ]:
block_raw = pd.read_csv(RAW_FOLDER + "/forms/block_survey.csv")
weighted_raw = pd.read_csv(RAW_FOLDER + "/forms/weighted_tlx.csv")
screening_raw = pd.read_csv(RAW_FOLDER + "/forms/screening.csv")

print("block survey:   ", block_raw.shape)
print("weighted survey:", weighted_raw.shape)
print("screening:      ", screening_raw.shape)

## 2. Clean each source separately
### 2.1 `main_data` (one row per trial)
The order matters: **keep the right sessions, remove exact duplicates, then check the structure**.
We remove duplicates *before* checking the structure, otherwise a duplicated row would look like a 25th trial.

In [ ]:
main = main_raw.copy()

# 1) keep only sessions of the right mode (this keeps pilot / test runs out of the real analysis)
before = len(main)
main = main[main["mode"] == MODE]
note("main_data", "rows removed because of a different mode", before - len(main))

# 2) remove rows that are exact copies of another row
n_duplicates = main.duplicated().sum()
main = main.drop_duplicates()
note("main_data", "exact duplicate rows removed", n_duplicates)

# 3) the controller writes true/false as text. Turn them into 1 / 0, and keep blanks as blanks (missing).
#    (A blank must stay missing: it means "not measured", which is different from 0 / False.)
true_false = {True: 1.0, False: 0.0, "true": 1.0, "false": 0.0, "True": 1.0, "False": 0.0}
main["first_selection_correct"] = main["first_selection_correct"].map(true_false)
main["no_selection"] = main["no_selection"].map(true_false)
main["technical_loss"] = main["technical_loss"].map(true_false)

# 4) tidy participant codes (remove stray spaces, capital letters)
main["participant"] = main["participant"].str.strip().str.upper()

**Structure check.** The study design promises 24 trials per participant, 6 in each of the 4 conditions, and each (participant, block, trial) exactly once.
`assert` makes the notebook stop right here if the data break that promise.

In [ ]:
trials_per_participant = main.groupby("participant").size()
trials_per_condition = main.groupby(["participant", "condition"]).size()

print("participants:", main["participant"].nunique())
print("trials per participant:", sorted(trials_per_participant.unique()))
print("trials per participant per condition:", sorted(trials_per_condition.unique()))

assert set(main["participant"]) == set(ROSTER), "the participants found are not the participants we expected"
assert (trials_per_participant == 24).all(), "someone does not have exactly 24 trials"
assert (trials_per_condition == TRIALS_PER_BLOCK).all(), "someone does not have 6 trials in every condition"
assert not main.duplicated(["participant", "block", "trial"]).any(), "the same trial appears twice"
print("Structure OK: 16 participants x 24 trials =", len(main), "trials")

**Missing values.** A blank is only a problem if it is *unexpected*. By design, a trial that timed out or was interrupted has no retrieval time:
the blank means "not measured", not "data lost". So we only expect complete performance values for trials that were **completed**.

In [ ]:
print("Trials by outcome:")
print(main["controller_status"].value_counts())

completed = main["controller_status"] == "completed_auto"
performance_columns = ["retrieval_time_s", "total_tab_switches", "incorrect_selections", "first_selection_correct"]

print()
print("Missing values among COMPLETED trials (should all be 0):")
print(main.loc[completed, performance_columns].isna().sum())
assert main.loc[completed, performance_columns].isna().sum().sum() == 0

# a completed trial has exactly one correct selection, so tab switches = incorrect selections + 1
completed_rows = main[completed]
assert (completed_rows["total_tab_switches"] == completed_rows["incorrect_selections"] + 1).all()
assert completed_rows["retrieval_time_s"].between(0, 20).all()      # the trial limit is 20 seconds
print("Completed trials are consistent.")

**Flags instead of deleting.** Timeouts and interrupted trials stay in the data, but we mark them.
This matters for the analysis: retrieval time is only analysed for completed trials, but a timeout is itself a result (the participant failed within 20 seconds),
so we want to be able to count them per condition.

In [ ]:
main["completed"] = main["controller_status"] == "completed_auto"
main["timed_out"] = main["controller_status"] == "timeout"
main["interrupted"] = main["controller_status"] == "interrupted"

note("main_data", "timeout trials (no retrieval time by design)", main["timed_out"].sum())
note("main_data", "interrupted trials", main["interrupted"].sum())

# drop columns we do not need further on (they are still in main_raw if we ever need them)
main_clean = main.drop(columns=["session_id", "mode", "measurement_source", "target_title", "notes"])
print("main_clean:", main_clean.shape)
main_clean.head(3)

### 2.2 Block NASA-TLX (one row per participant per condition)
After each condition, participants rated six aspects of workload. What we fix:
- the Google Form's very long question headings become short column names;
- the participant code is tidied (spaces, capitals);
- the condition text (for example `V-H: Vertical High`) becomes the code `V-H`, so it matches the main data;
- the ratings are 0-10 on the form, and we multiply them by 10 so they are on a 0-100 scale (this is the agreed rescaling);
- duplicate submissions are resolved, and we check that every participant rated all 4 conditions.

In [ ]:
block = block_raw.copy()

# short names for the 11 columns, in the order they appear in the form export
block.columns = ["timestamp", "score", "participant", "condition_label"] + DIMS + ["comment"]
block = block.drop(columns=["score"])

# tidy participant codes, and report how many had to be fixed
codes_before = block["participant"].copy()
block["participant"] = block["participant"].str.strip().str.upper()
note("block survey", "participant codes that needed tidying", (codes_before != block["participant"]).sum())

# "V-H: Vertical High"  ->  "V-H"   (keep the text before the colon)
block["condition"] = block["condition_label"].str.split(":").str[0].str.strip()
block = block.drop(columns=["condition_label"])
assert set(block["condition"]) <= set(CONDITIONS)

# a code that is not on our list is probably a typo, and needs a human to look at it
unknown_codes = set(block["participant"]) - set(ROSTER)
note("block survey", "participant codes not on the roster", len(unknown_codes))
assert len(unknown_codes) == 0, unknown_codes

# ratings are 0-10 on the form -> multiply by 10 to get 0-100
assert block[DIMS].min().min() >= 0 and block[DIMS].max().max() <= 10, "a rating is outside 0-10"
block[DIMS] = block[DIMS] * 10
block["timestamp"] = pd.to_datetime(block["timestamp"], format="%m/%d/%Y %H:%M:%S")

In [ ]:
# if someone submitted the same condition twice, keep the LATEST submission
n_before = len(block)
block = block.sort_values("timestamp").drop_duplicates(["participant", "condition"], keep="last")
note("block survey", "duplicate submissions removed (kept the latest)", n_before - len(block))

# expect 16 participants x 4 conditions = 64 rows
assert len(block) == N_PARTICIPANTS * len(CONDITIONS), len(block)

# "raw TLX" = the simple average of the six ratings (no weights)
block["raw_tlx"] = block[DIMS].mean(axis=1)

block_clean = block.drop(columns=["comment"]).sort_values(["participant", "condition"]).reset_index(drop=True)
print("block_clean:", block_clean.shape)
block_clean.head(3)

### 2.3 Weighted NASA-TLX (one row per participant)
Each participant answered 15 questions of the form "which mattered more for the task: A or B?".
A dimension's **weight is the number of times it was chosen** (between 0 and 5). The six weights must add up to 15, because there are 15 questions.
We later use these weights to turn the six block ratings into one weighted workload score.

If a participant did not submit this form, they simply have no row. We report that and **do not invent values** for them.

In [ ]:
weighted = weighted_raw.copy()

# short names: the 15 question columns become pair_1 ... pair_15
weighted.columns = ["timestamp", "score", "participant"] + ["pair_" + str(i) for i in range(1, 16)]
weighted["participant"] = weighted["participant"].str.strip().str.upper()

# each answer is the name of the dimension that was chosen. Its first word identifies it:
# "Mental Demand" -> "Mental", "Performance Demand" -> "Performance", "Frustration" -> "Frustration"
first_word_to_dimension = {"Mental": "mental_demand", "Physical": "physical_demand", "Temporal": "temporal_demand",
                           "Performance": "performance", "Effort": "effort", "Frustration": "frustration"}
pair_columns = ["pair_" + str(i) for i in range(1, 16)]
chosen = weighted[pair_columns].apply(lambda column: column.str.split().str[0].map(first_word_to_dimension))
assert chosen.notna().all().all(), "an answer was not recognised"

# weight of a dimension = how many of the 15 answers chose it
weights = pd.DataFrame({"participant": weighted["participant"]})
for dim in DIMS:
    weights["w_" + dim] = (chosen == dim).sum(axis=1)

assert (weights[["w_" + d for d in DIMS]].sum(axis=1) == 15).all(), "the weights of a participant must add up to 15"
assert not weights["participant"].duplicated().any(), "someone submitted this form twice"

missing_form = sorted(set(ROSTER) - set(weights["participant"]))
note("weighted survey", "participants who did not submit it " + str(missing_form), len(missing_form))

weights_clean = weights.sort_values("participant").reset_index(drop=True)
print("weights_clean:", weights_clean.shape)
weights_clean.head(3)

### 2.4 Screening / background form (one row per person)
Two things matter here:
- **Privacy.** The form collects names. We drop the name column immediately so names never travel into the files we analyse.
- Some people filled in the screening form but were never enrolled in the study. They have no participant code, so we remove them and record how many.

In [ ]:
screening = screening_raw.copy()

screening.columns = ["timestamp", "consent", "name", "age", "uses_browser_on_desktop", "desktop_browsing_frequency",
                     "chrome_is_main_browser", "multi_tab_frequency", "window_habit", "usual_tab_orientation", "participant"]
screening = screening.drop(columns=["name"])          # personal data: removed right away

screening["participant"] = screening["participant"].str.strip().str.upper()

not_enrolled = screening["participant"].isna()
note("screening", "responses without a participant code (never enrolled)", not_enrolled.sum())
screening = screening[~not_enrolled]

assert not screening["participant"].duplicated().any(), "someone was screened twice"
assert set(screening["participant"]) == set(ROSTER)
assert screening["consent"].str.contains("agree", case=False).all(), "someone did not consent"

screening_clean = screening.drop(columns=["timestamp", "consent"]).sort_values("participant").reset_index(drop=True)
print("screening_clean:", screening_clean.shape)
screening_clean.head(3)

## 3. Who is present in which source?
Before merging, we check that every participant appears in every source. If someone is missing from a form, we want to see it here,
not discover it later as a mysterious blank in the results.

In [ ]:
coverage = pd.DataFrame({"participant": ROSTER})
coverage["main_data_trials"] = coverage["participant"].map(main_clean.groupby("participant").size())
coverage["block_ratings"] = coverage["participant"].map(block_clean.groupby("participant").size())
coverage["has_weighted_form"] = coverage["participant"].isin(weights_clean["participant"])
coverage["has_screening"] = coverage["participant"].isin(screening_clean["participant"])
coverage

## 4. Combine into two analysis tables
The statistics in the paper work at two levels, so we build two tables:

- **`trial_level`** (384 rows): one row per trial. Used for accuracy, which is a yes/no result on each trial. Participant background is attached.
- **`condition_level`** (64 rows): one row per participant per condition, with the trial results summarised, plus that condition's block TLX, the participant's weights, and their background.
  Used for retrieval time and weighted TLX, which the paper analyses with one value per participant per condition.

We always merge on **participant** (and **condition**), and after each merge we check that the number of rows did not change by accident.

In [ ]:
# ---- trial_level: every trial, plus the participant's background ----
trial_level = main_clean.merge(screening_clean, on="participant", how="left")
assert len(trial_level) == 384, "the merge changed the number of trials"
trial_level = trial_level.sort_values(["participant", "block", "trial"]).reset_index(drop=True)
print("trial_level:", trial_level.shape)

For `condition_level` we first **summarise the trials** of each participant in each condition (6 trials become one row). The rules follow Section 4.7 of the paper:

- **Retrieval time** and **incorrect selections** use **completed trials only**.
- **First-selection accuracy** uses every trial where a first selection could be observed (so a timeout where the participant did click something still counts, as an incorrect first selection).
  Trials with no observable first selection are left out of accuracy and are counted separately.
- **Timeouts** are counted, so they can be reported next to the averages.

In [ ]:
# helper columns: a value only for completed trials (blank otherwise), so a plain mean uses completed trials only
main_clean["rt_if_completed"] = main_clean["retrieval_time_s"].where(main_clean["completed"])
main_clean["incorrect_if_completed"] = main_clean["incorrect_selections"].where(main_clean["completed"])
main_clean["no_selection_flag"] = main_clean["no_selection"] == 1

summary = main_clean.groupby(["participant", "condition", "orientation", "tab_load"]).agg(
    n_trials=("trial", "count"),
    n_completed=("completed", "sum"),
    n_timeouts=("timed_out", "sum"),
    n_interrupted=("interrupted", "sum"),
    n_no_selection=("no_selection_flag", "sum"),
    n_accuracy_valid=("first_selection_correct", "count"),          # trials where a first selection was observed
    mean_rt_s=("rt_if_completed", "mean"),
    mean_incorrect_selections=("incorrect_if_completed", "mean"),
    first_try_accuracy=("first_selection_correct", "mean"),          # mean of 1/0 = proportion correct
).reset_index()
assert len(summary) == 64
summary.head(4)

Now we add the survey data. The block ratings match on **participant and condition**; the weights and the background match on **participant**.

The **weighted NASA-TLX** for a condition is: (weight of each dimension x that dimension's rating), added up for the six dimensions, divided by 15.
A participant without a weighting form has no weights, so their weighted TLX stays blank (it is not guessed).

In [ ]:
condition_level = summary.merge(block_clean.drop(columns=["timestamp"]), on=["participant", "condition"], how="left")
condition_level = condition_level.merge(weights_clean, on="participant", how="left")
condition_level = condition_level.merge(screening_clean, on="participant", how="left")
assert len(condition_level) == 64, "a merge changed the number of rows"

# weighted TLX = (sum of weight x rating over the six dimensions) / 15
condition_level["weighted_tlx"] = sum(condition_level["w_" + d] * condition_level[d] for d in DIMS) / 15

condition_level = condition_level.sort_values(["participant", "condition"]).reset_index(drop=True)
print("condition_level:", condition_level.shape)
condition_level.head(4)

## 5. Final checks and save
A last set of checks on the two finished tables, then we save them (and the cleaning log) as CSV files for the next notebooks.

In [ ]:
assert (trial_level.groupby("participant").size() == 24).all()
assert (condition_level.groupby("participant").size() == 4).all()
assert (condition_level["n_trials"] == TRIALS_PER_BLOCK).all()
assert "name" not in trial_level.columns        # no personal data in the outputs

print("Missing values left in condition_level (expected: weighted TLX for people without the weighting form):")
missing = condition_level.isna().sum()
print(missing[missing > 0])

os.makedirs(OUT_FOLDER, exist_ok=True)
trial_level.to_csv(OUT_FOLDER + "/trial_level.csv", index=False)
condition_level.to_csv(OUT_FOLDER + "/condition_level.csv", index=False)
cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(OUT_FOLDER + "/cleaning_log.csv", index=False)
print()
print("Saved trial_level.csv, condition_level.csv and cleaning_log.csv to", OUT_FOLDER)
cleaning_log

**What the cleaning log tells the reader:** every row above is something we found and handled, with a count. Nothing was changed silently.
The cleaned tables are now ready for the next notebooks: `analysis_01` (description and plots) and `analysis_02_inference` (the statistical tests).